In [ ]:
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from procompa import get_project_root, get_data_dir

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

## YeastMap data

In [ ]:
data_dir_y = get_data_dir() / "26.03_yeast.MAP"
prefix = "complex_portal_scerevisiae_559292_reduced_complexes_20251007."

for path in data_dir_y.glob(f"{prefix}*.txt"):
    globals()[path.stem.replace(prefix, "")] = (
        pl.read_csv(path, separator="\n", has_header=False, new_columns=["raw"])
        .with_columns(
            pl.col("raw")
            .str.replace_all("\t", " ")
            .str.split(" ")
            .alias("_proteins")
        )
        .with_columns(
            pl.col("_proteins").alias("all_proteins"),        # full list of IDs
            pl.col("_proteins").list.len().alias("n_proteins"), # complex size
        )
        .pipe(lambda df: df.with_columns([
            pl.col("_proteins")
            .list.get(i, null_on_oob=True)
            .alias(f"protein{i + 1}")
            for i in range(df["_proteins"].list.len().max())
        ]))
        .drop(["raw", "_proteins"])
    )

In [ ]:
# ── 1. Complex-level overlap ──
train_sets = set(train["all_proteins"].map_elements(frozenset, return_dtype=pl.Object))
test_sets  = set(test["all_proteins"].map_elements(frozenset, return_dtype=pl.Object))
print("Identical complexes in train∩test:", len(train_sets & test_sets))

# ── 2. Protein-level overlap ──
train_proteins = set(p for row in train["all_proteins"].to_list() for p in row)
test_proteins  = set(p for row in test["all_proteins"].to_list()  for p in row)
print("Proteins in train∩test:", len(train_proteins & test_proteins))

# ── 3. PPI-level overlap ──
def ppi_set(df):
    return set(
        frozenset([a, b])
        for a, b in zip(df["protein1"].to_list(), df["protein2"].to_list())
    )

train_pairs     = ppi_set(train_ppis)
test_pairs      = ppi_set(test_ppis)
neg_train_pairs = ppi_set(neg_train_ppis)
neg_test_pairs  = ppi_set(neg_test_ppis)

print("Pos train∩test PPIs:",        len(train_pairs & test_pairs))
print("Neg train∩neg test PPIs:",    len(neg_train_pairs & neg_test_pairs))
print("Pos train ∩ neg train PPIs:", len(train_pairs & neg_train_pairs))  # should be 0!
print("Pos test  ∩ neg test PPIs:",  len(test_pairs  & neg_test_pairs))   # should be 0!
print("Pos train ∩ neg test PPIs:",  len(train_pairs & neg_test_pairs))
print("Pos test  ∩ neg train PPIs:", len(test_pairs  & neg_train_pairs))

In [ ]:
for name, obj in list(globals().items()):
    if isinstance(obj, pl.DataFrame):
        print(f"{name:20s}: {len(obj):>6,} rows")

In [ ]:
# Sum of all pairwise combinations from train complexes should equal len(train_ppis)
import math
train.with_columns(
    pl.col("n_proteins").map_elements(lambda n: math.comb(n, 2), return_dtype=pl.Int64)
).select("n_proteins").sum()
# if this equals 2081, the above is confirmed

Yeastmap prediction

In [ ]:
YM_pred = pl.read_csv(
    data_dir/"Complex_Portal/YeastMap/yeastmap.pairsWprob",
    separator="\t",
    infer_schema_length=10000,
)

In [ ]:
YM_pred.height

In [ ]:
YM_pred_comple = pl.read_csv(
    get_data_dir() / "26.03_yeast.MAP/yeast.MAP_complexes_wConfidenceScores_wGenenames_total779_20251214.csv",
    separator=",",
    infer_schema_length=10000,  
    truncate_ragged_lines=True,
)

In [ ]:
YM_pred_comple.height

In [ ]:
#How many proteins make up the predicted complexes

YM_pred_comple = YM_pred_comple.with_columns(
    pl.col("UniProt_ACCs").str.split(" ").list.len().alias("n_subcomplexes")
)

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))

sizes = YM_pred_comple["n_subcomplexes"].to_numpy()
median_val = YM_pred_comple["n_subcomplexes"].median()
max_val = YM_pred_comple["n_subcomplexes"].max()
total_complexes = len(YM_pred_comple)


n, bins, patches = plt.hist(
    sizes, 
    bins=25, 
    color="#2b7bba", 
    edgecolor="white", 
    alpha=0.85, 
    log=False
)


stats_text = (
    f"Total Complexes: {total_complexes:,}\n"
    f"Largest Complex: {max_val:,}"
)

plt.gca().text(
    0.95, 0.9, stats_text,
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="0.8", alpha=0.9)
)

plt.title("Distribution of Protein Subunit Counts in Predicted YeastMap Complexes", pad=20, fontsize=16)
plt.xlabel("Number of Proteins (Subunits) per Complex", labelpad=12)
plt.ylabel("Frequency", labelpad=12)


sns.despine(left=True, bottom=True)
plt.legend(loc="upper right", bbox_to_anchor=(0.95, 0.95), frameon=False)
plt.tight_layout()

plt.show()

In [ ]:
plot_data = YM_pred_comple.with_columns(
    pl.when(pl.col("n_subcomplexes") > 10)
    .then(pl.lit("10+"))

    .otherwise(pl.col("n_subcomplexes").cast(pl.Utf8))
    .alias("display_size")
)

df_pd = plot_data.to_pandas()
order = [str(i) for i in range(2, 11)] + ["10+"]

plt.figure(figsize=(8, 6))
sns.set_theme(style="whitegrid")

sns.countplot(data=df_pd, x="display_size", order=order, color="#4a90e2", alpha=0.9)
plt.grid(True, which="both", axis="both")
plt.title("Exact Protein Counts per Predicted Complex", pad=20)
plt.xlabel("Exact Number of Subunits")
plt.ylabel("Frequency")
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

## IntAct yeast data

In [ ]:
IntAct_yeast = pl.read_csv(
    data_dir / "IntAct/yeast_miTab_2.7.txt",
    separator="\t",
    has_header=True,
    ignore_errors=True,
    infer_schema_length=0,

)


In [ ]:
len(IntAct_yeast.columns)

filter 
Host organism(s): taxid:559292(yeast)|taxid:559292(Saccharomyces cerevisiae) #there should only be this related to yeast
Type(s) interactor B for psi-mi:"MI:0326"(protein)

In [ ]:
IntAct_yeast_neg = pl.read_csv(
    data_dir / "IntAct/yeast_negative_miTab_2.7.txt",
    separator="\t",
    has_header=True,
    ignore_errors=True,
    infer_schema_length=0,
)
IntAct_yeast_neg.height
print(f"the IntAct yeast df contains {IntAct_yeast_neg.height} entries")

# the IntAct yeast df contains 7 entries


In [ ]:
IntAct_yeast.height
print(f"the IntAct yeast df contains {IntAct_yeast.height} entries")

In [ ]:
yeast = "taxid:559292"

Yeast_yeastInteract = IntAct_yeast.filter(
    pl.col("Taxid interactor A").str.contains(yeast) &
    pl.col("Taxid interactor B").str.contains(yeast)
)

In [ ]:
non_uniprot_ids = (
    Yeast_yeastInteract
    .select([
        pl.col("#ID(s) interactor A"),
        pl.col("ID(s) interactor B"),
    ])
    .unpivot(value_name="id")
    .filter(~pl.col("id").str.starts_with("uniprot"))
    .select("id")
    .unique()
)

In [ ]:
non_uniprot_ids_list = non_uniprot_ids["id"].to_list()
with open("non_uniprot_ids.txt", "w") as f:
    for item in non_uniprot_ids_list:
        f.write(f"{item}\n")

## Stoic 

In [ ]:
stoic = pl.read_csv(data_dir / "Stoic/data_file_stoic.csv")

benchmark stoic

In [ ]:
'''
train_stoic = stoic.filter(pl.col("split") == "train")
train_stoic.write_csv(data_dir / "Stoic/train_stoic.csv", include_header=True)
ids_train = train_stoic.select("pdb_id").to_series().to_list()
with open(data_dir / "Stoic/train_ids_stoic.txt", "w") as f:
    f.write("\n".join(ids_train))
'''

look for overlap between complex and stoic trainiing - all species

In [ ]:
stoic_dir = data_dir / "Stoic/all_species"

files = sorted(stoic_dir.glob("*.tsv"))

    
complex_all_species = pl.concat(
    [
        pl.read_csv(f, separator="\t", has_header=True, truncate_ragged_lines=True, quote_char=None)
        .with_columns(pl.lit(f.stem).alias("source_file"))
        for f in files
    ],
    how="vertical",
)
complex_all_species.head(2)

In [ ]:
complex_all_species_known_stoichiometry = complex_all_species.select(
    ["#Complex ac", "Identifiers (and stoichiometry) of molecules in complex", "source_file"]
).filter(~pl.col("Identifiers (and stoichiometry) of molecules in complex").str.contains(r"\(0\)")).with_columns(
        pl.col("Identifiers (and stoichiometry) of molecules in complex")
        .str.split("|")                         
        .list.eval(
            pl.element().str.extract(r"^([A-Z0-9-]+)")
        )
        .alias("clean_ids")
    )

complex_all_species_known_stoichiometry.head(2)

In [ ]:
mapped_stoic_ids = pl.read_csv(data_dir / "Stoic/idmapping_2026_05_20.tsv", separator="\t", has_header=True)
mapped_stoic_id_list = mapped_stoic_ids["To"].unique().to_list()

In [ ]:
complex_all_species_known_stoichiometry = (
    complex_all_species_known_stoichiometry.with_columns(
        pl.col("clean_ids").list.len().alias("has_n"),
        
        pl.col("clean_ids")
        .list.eval(pl.element().is_in(mapped_stoic_id_list))
        .list.sum()
        .alias("n_in_stoic_training"),
    )
)

In [ ]:
SOURCE_FILTER = "559292"  

filter_expr = pl.col("source_file") == SOURCE_FILTER if SOURCE_FILTER is not None else pl.lit(True)

plot_data = complex_all_species_known_stoichiometry.filter(filter_expr).with_columns(
    pl.when(pl.col("has_n") >= 7).then(pl.lit("7+")).otherwise(pl.col("has_n").cast(pl.Utf8)).alias("display_size"),
    pl.when(pl.col("n_in_stoic_training") >= 7).then(pl.lit("7+")).otherwise(pl.col("n_in_stoic_training").cast(pl.Utf8)).alias("mapped_display")
)

df_pd = plot_data.to_pandas()

min_size = int(df_pd["has_n"].min()) if not df_pd.empty else 1
x_order = [str(i) for i in range(min_size, 7)] + ["7+"]
stack_order = [str(i) for i in range(0, 7)] + ["7+"]

pivot_df = df_pd.groupby(["display_size", "mapped_display"]).size().unstack(fill_value=0)
pivot_df = pivot_df.reindex(index=x_order, columns=stack_order, fill_value=0)

group_totals = pivot_df.sum(axis=1)
new_x_labels = [f"{idx}\n(n={group_totals[idx]})" for idx in pivot_df.index]

stack_totals = pivot_df.sum(axis=0)
new_legend_labels = [f"{col} (n={stack_totals[col]})" for col in pivot_df.columns]

plt.figure(figsize=(9, 7))
sns.set_theme(style="whitegrid")
ax = plt.gca()

pivot_df.plot(
    kind="bar", stacked=True, ax=ax, cmap="YlGnBu", edgecolor="#444444", linewidth=0.5, alpha=0.9, width=0.75
)

for rect in ax.patches:
    height = rect.get_height()
    width = rect.get_width()
    x = rect.get_x()
    y = rect.get_y()
    
    if height > 0:
        label_x = x + width / 2
        label_y = y + height / 2
        text_color = "black" 
        ax.text(label_x, label_y, f"{int(height)}", ha="center", va="center", fontsize=7, color=text_color)

title_suffix = f"- {SOURCE_FILTER}" if SOURCE_FILTER else "(All Species)"
plt.title(f"Protein Counts per Complex grouped by Mapped IDs {title_suffix}", pad=20, fontsize=14 )
plt.xlabel("Unique Subunits in Complex (Capped at 7+)", labelpad=12, fontsize=11)
plt.ylabel("Frequency", labelpad=12, fontsize=11)
ax.set_xticklabels(new_x_labels, rotation=0, fontsize=10)

handles, _ = ax.get_legend_handles_labels()
plt.legend(handles, new_legend_labels, title="Mapped IDs in Training Set", loc="upper right", frameon=True)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

overlap between stoic training and its benchmark

In [ ]:
train_sequences_nested = (
    stoic.filter(pl.col("split") == "train")
    .select("seq_name")
    .drop_nulls()
    .unique()
    .to_series()
    .to_list()
)

train_sequences = []
for s in train_sequences_nested:
    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1]
    train_sequences.extend([item.strip() for item in s.split(";") if item.strip()])
train_sequences = set(train_sequences)

def count_overlap(seq_str, train_set):
    if not seq_str:
        return "0"
    if seq_str.startswith("(") and seq_str.endswith(")"):
        seq_str = seq_str[1:-1]
    individual_seqs = [item.strip() for item in seq_str.split(";") if item.strip()]
    
    overlap_count = sum(1 for s in individual_seqs if s in train_set)
    return "7+" if overlap_count >= 7 else str(overlap_count)

plot_data = (
    stoic.filter(pl.col("split") == "benchmark")
    .with_columns(
        pl.when(pl.col("num_nodes") >= 7).then(pl.lit("7+")).otherwise(pl.col("num_nodes").cast(pl.Utf8)).alias("display_size"),
        pl.col("seq_name")
        .map_elements(lambda x: count_overlap(x, train_sequences), return_dtype=pl.Utf8)
        .alias("mapped_display")
    )
)

df_pd = plot_data.to_pandas()

min_size = int(df_pd["num_nodes"].min()) if not df_pd.empty else 1
x_order = [str(i) for i in range(min_size, 7)] + ["7+"]
stack_order = [str(i) for i in range(0, 7)] + ["7+"]

pivot_df = df_pd.groupby(["display_size", "mapped_display"]).size().unstack(fill_value=0)
pivot_df = pivot_df.reindex(index=x_order, columns=stack_order, fill_value=0)

group_totals = pivot_df.sum(axis=1)
new_x_labels = [f"{idx}\n(n={group_totals[idx]})" for idx in pivot_df.index]

stack_totals = pivot_df.sum(axis=0)
new_legend_labels = [f"{col} (n={stack_totals[col]})" for col in pivot_df.columns]

plt.figure(figsize=(11, 7))
sns.set_theme(style="whitegrid")
ax = plt.gca()

pivot_df.plot(
    kind="bar", stacked=True, ax=ax, cmap="YlGnBu", edgecolor="#444444", linewidth=0.5, alpha=0.9, width=0.75
)

for rect in ax.patches:
    height = rect.get_height()
    width = rect.get_width()
    x = rect.get_x()
    y = rect.get_y()
    
    if height > 0:
        label_x = x + width / 2
        label_y = y + height / 2
        text_color = "black"
        ax.text(label_x, label_y, f"{int(height)}", ha="center", va="center", fontsize=7, color=text_color)

plt.title("Benchmark Complex Counts grouped by Number of Overlapping Sequences in Training Split", pad=20, fontsize=14)
plt.xlabel("Number of Nodes (Capped at 7+)", labelpad=12, fontsize=11)
plt.ylabel("Frequency", labelpad=12, fontsize=11)
ax.set_xticklabels(new_x_labels, rotation=0, fontsize=10)

handles, _ = ax.get_legend_handles_labels()
plt.legend(handles, new_legend_labels, title="Mapped Sequences in Training", loc="upper right", frameon=True)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

In [ ]:
#check overlap of sequences
print (f"Unique train sequences: {len(train_sequences)}")
benchmark_sequences_nested = (
    stoic.filter(pl.col("split") == "benchmark")
    .select("seq_name")
    .drop_nulls()
    .unique()
    .to_series()
    .to_list()
)

benchmark_sequences = []
for s in benchmark_sequences_nested:
    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1]
    benchmark_sequences.extend([item.strip() for item in s.split(";") if item.strip()])

unique_benchmark = set(benchmark_sequences)
overlapping_unique = unique_benchmark.intersection(train_sequences)

print(f"Unique benchmark sequences: {len(unique_benchmark)}")
print(f"Overlapping unique sequences: {len(overlapping_unique)}")

In [ ]:
train_clusters_nested = (
    stoic.filter(pl.col("split") == "train")
    .select("cluster_label")
    .drop_nulls()
    .unique()
    .to_series()
    .to_list()
)

train_clusters = []
for c in train_clusters_nested:
    c_str = str(c)
    if c_str.startswith("(") and c_str.endswith(")"):
        c_str = c_str[1:-1]
    train_clusters.extend([item.strip() for item in c_str.split(";") if item.strip()])
train_clusters = set(train_clusters)

def count_cluster_overlap(cluster_str, train_set):
    if not cluster_str:
        return "0"
    cluster_str = str(cluster_str)
    if cluster_str.startswith("(") and cluster_str.endswith(")"):
        cluster_str = cluster_str[1:-1]
    individual_clusters = [item.strip() for item in cluster_str.split(";") if item.strip()]
    
    overlap_count = sum(1 for c in individual_clusters if c in train_set)
    return "7+" if overlap_count >= 7 else str(overlap_count)

plot_data = (
    stoic.filter(pl.col("split") == "benchmark")
    .with_columns(
        pl.when(pl.col("num_nodes") >= 7).then(pl.lit("7+")).otherwise(pl.col("num_nodes").cast(pl.Utf8)).alias("display_size"),
        pl.col("cluster_label")
        .map_elements(lambda x: count_cluster_overlap(x, train_clusters), return_dtype=pl.Utf8)
        .alias("mapped_display")
    )
)

df_pd = plot_data.to_pandas()

min_size = int(df_pd["num_nodes"].min()) if not df_pd.empty else 1
x_order = [str(i) for i in range(min_size, 7)] + ["7+"]
stack_order = [str(i) for i in range(0, 7)] + ["7+"]

pivot_df = df_pd.groupby(["display_size", "mapped_display"]).size().unstack(fill_value=0)
pivot_df = pivot_df.reindex(index=x_order, columns=stack_order, fill_value=0)

group_totals = pivot_df.sum(axis=1)
new_x_labels = [f"{idx}\n(n={group_totals[idx]})" for idx in pivot_df.index]

stack_totals = pivot_df.sum(axis=0)
new_legend_labels = [f"{col} (n={stack_totals[col]})" for col in pivot_df.columns]

plt.figure(figsize=(11, 7))
sns.set_theme(style="whitegrid")
ax = plt.gca()

pivot_df.plot(
    kind="bar", stacked=True, ax=ax, cmap="YlGnBu", edgecolor="#444444", linewidth=0.5, alpha=0.9, width=0.75
)

for rect in ax.patches:
    height = rect.get_height()
    width = rect.get_width()
    x = rect.get_x()
    y = rect.get_y()
    
    if height > 0:
        label_x = x + width / 2
        label_y = y + height / 2
        text_color = "black" if y < (group_totals.max() * 0.4) else "white"
        ax.text(label_x, label_y, f"{int(height)}", ha="center", va="center", fontsize=7, color=text_color, weight="bold")

plt.title("Benchmark Complex Counts grouped by Number of Overlapping Clusters in Training Split", pad=20, fontsize=14, weight="bold")
plt.xlabel("Number of Nodes (Capped at 7+)", labelpad=12, fontsize=11)
plt.ylabel("Frequency", labelpad=12, fontsize=11)
ax.set_xticklabels(new_x_labels, rotation=0, fontsize=10)

handles, _ = ax.get_legend_handles_labels()
plt.legend(handles, new_legend_labels, title="Mapped Clusters in Training", loc="upper right", frameon=True)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

can i recretae dataset to ahve 1350 complexes if i filter all benchmakr complexes so that at least 1